# Deep dive 3/13: `branches`

Only 350 rows -- small enough to eyeball directly (section 0 dumps the whole table), but still checked column-by-column since a small table is exactly where a subtle bug hides easiest (few rows means low odds of it standing out in a spot check).

Source of truth for what Silver *should* do: `data_pipelines/silver/table_specs.py`'s `branches_spec`. Source of truth for what the data *actually* looks like: this notebook, run against your real `data/latam_bank.duckdb`. Data dictionary declares 350 rows, Source: Internal, Partition: full_snapshot.

Run top to bottom. Checklist near the end; free cells after that.

In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 0. The whole table

350 rows is small enough to just look at directly. Worth a manual scan before the structured checks below -- sometimes a wrong value jumps out faster to a human eye than to a query you didn't think to write.

In [2]:
q_full("SELECT * FROM silver.dim_branches ORDER BY branch_id")

,branch_id,branch_code,branch_name,branch_type,address,city,state,country,postal_code,geographic_zone,phone,email,opening_time,closing_time,has_atms,atm_count,has_teller_windows,teller_window_count,latitude,longitude,branch_opening_date,branch_status
0,SUC-01J639PQ,S0185,Banco LATAM Cali Centro,Express,"Circular Avenida Caracas #504, Barrio San Fernando",Cali,Valle del Cauca,Colombia,762627,Urbana,+57 5 322 6601,sucursal0185@bancolatam.com,09:30:00,18:00:00,True,8,True,10,3.551193,-76.539263,1991-06-04,Active
1,SUC-033KHYS2,S0144,Banco LATAM Puebla Plaza,Premium,"Camino 5 de Mayo 3, Centro",Puebla,Puebla,México,72000,Urbana,+54 69 5850 5327,sucursal0144@bancolatam.com,09:30:00,17:00:00,True,2,True,5,0.074828,0.070450,2006-02-27,Active
2,SUC-06H3RFFA,S0330,Banco LATAM Córdoba Oeste,Express,"Camino Avenida Belgrano 716, Jardín",Córdoba,Córdoba,Argentina,X5000,Urbana,+54 34 3200 4244,sucursal0330@bancolatam.com,09:30:00,17:30:00,True,6,True,4,-31.429968,-64.125926,1998-04-30,Active
3,SUC-06LBAMRN,S0257,Banco LATAM Medellín 257,Express,"Calle 756 #9-73, Barrio El Poblado",Medellín,Antioquia,Colombia,51389,Urbana,+57 5 602 9270,sucursal0257@bancolatam.com,09:00:00,20:00:00,True,4,True,11,6.256924,-75.475284,1991-06-02,Active
4,SUC-07AK38PR,S0268,Banco LATAM Medellín 268,Corporate,"Circular Carrera 30 #811, Barrio Buenos Aires",Medellín,Antioquia,Colombia,50279,Urbana,+57 4 562 8837,sucursal0268@bancolatam.com,08:30:00,18:00:00,True,3,True,12,6.312954,-75.505894,1998-11-29,Active
5,SUC-0KBWS25U,S0212,Banco LATAM Cali 212,Premium,"Avenida Carrera 50 #695, Barrio Tequendama",Cali,Valle del Cauca,Colombia,764390,Urbana,+57 2 684 7382,sucursal0212@bancolatam.com,08:30:00,20:00:00,True,2,True,7,3.492653,-76.569100,1991-06-03,Active
6,SUC-0LZRXHCF,S0332,Banco LATAM Buenos Aires Plaza,Corporate,"Avenida Avenida Callao 535, Colegiales",Buenos Aires,Ciudad Autónoma de Buenos Aires,Argentina,C1000,Urbana,+54 41 6918 7528,sucursal0332@bancolatam.com,09:30:00,20:00:00,True,2,True,8,-34.529085,-58.320346,1990-02-20,Active
7,SUC-0MPZUQLJ,S0281,Banco LATAM Rosario Centro,Express,"Diagonal Avenida Belgrano 53, Pellegrini",Rosario,Santa Fe,Argentina,S2000,Urbana,+54 73 9410 8993,sucursal0281@bancolatam.com,08:30:00,17:00:00,True,6,True,3,-32.959799,-60.687730,1992-11-06,Active
8,SUC-0N077NAM,S0322,Banco LATAM Córdoba Centro,Corporate,"Avenida Calle Lavalle 332, Cerro de las Rosas",Córdoba,Córdoba,Argentina,X5000,Urbana,+54 63 5940 9185,sucursal0322@bancolatam.com,08:00:00,20:00:00,True,8,True,12,-31.485808,-64.096993,2016-11-28,Temporarily Closed
9,SUC-0OJNMRLP,S0159,Banco LATAM Puebla Plaza,Corporate,"Andador México 324, Centro",Puebla,Puebla,México,72000,Urbana,+54 24 5748 5937,sucursal0159@bancolatam.com,08:00:00,18:00:00,True,8,True,10,-0.023717,0.064424,1991-05-20,Active


## 1. Column inventory

In [3]:
expected_silver_columns = [
    "branch_id", "branch_code", "branch_name", "branch_type", "address", "city", "state",
    "country", "postal_code", "geographic_zone", "phone", "email", "opening_time", "closing_time",
    "has_atms", "atm_count", "has_teller_windows", "teller_window_count", "latitude", "longitude",
    "branch_opening_date", "branch_status",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_branches'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.dim_branches:", missing or "none")
print("Extra in silver.dim_branches:", extra or "none")
print()
print("Column order as built:", actual_silver_columns)

Missing from silver.dim_branches: none
Extra in silver.dim_branches: none

Column order as built: ['branch_id', 'branch_code', 'branch_name', 'branch_type', 'address', 'city', 'state', 'country', 'postal_code', 'geographic_zone', 'phone', 'email', 'opening_time', 'closing_time', 'has_atms', 'atm_count', 'has_teller_windows', 'teller_window_count', 'latitude', 'longitude', 'branch_opening_date', 'branch_status']


In [4]:
print("Bronze columns:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='branches' ORDER BY ordinal_position
"""))

print("Silver columns:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_branches' ORDER BY ordinal_position
"""))

Bronze columns:


,column_name,data_type
0,branch_id,VARCHAR
1,branch_code,VARCHAR
2,branch_name,VARCHAR
3,branch_type,VARCHAR
4,address,VARCHAR
5,city,VARCHAR
6,state,VARCHAR
7,country,VARCHAR
8,postal_code,VARCHAR
9,geographic_zone,VARCHAR


Silver columns:


,column_name,data_type
0,branch_id,VARCHAR
1,branch_code,VARCHAR
2,branch_name,VARCHAR
3,branch_type,VARCHAR
4,address,VARCHAR
5,city,VARCHAR
6,state,VARCHAR
7,country,VARCHAR
8,postal_code,VARCHAR
9,geographic_zone,VARCHAR


## 2. Row-count reconciliation

In [5]:
bronze_total = q("SELECT count(*) AS n FROM bronze.branches")["n"][0]
bronze_distinct_pk = q("SELECT count(DISTINCT branch_id) AS n FROM bronze.branches")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.dim_branches")["n"][0]
silver_distinct_pk = q("SELECT count(DISTINCT branch_id) AS n FROM silver.dim_branches")["n"][0]

print(f"bronze.branches:      {bronze_total:,} rows, {bronze_distinct_pk:,} distinct branch_id")
print(f"silver.dim_branches:  {silver_total:,} rows, {silver_distinct_pk:,} distinct branch_id")
print()
print("Dictionary declares 350 rows -- matches" if bronze_total == 350 else f"Dictionary declares 350 rows -- MISMATCH, Bronze has {bronze_total}")
print("Duplicate raw rows in Bronze:", bronze_total - bronze_distinct_pk)
print("OK -- Silver row count matches Bronze distinct PK count" if silver_total == bronze_distinct_pk
      else "MISMATCH -- investigate")

bronze.branches:      350 rows, 350 distinct branch_id
silver.dim_branches:  350 rows, 350 distinct branch_id

Dictionary declares 350 rows -- matches
Duplicate raw rows in Bronze: 0
OK -- Silver row count matches Bronze distinct PK count


## 3. NOT NULL audit

Dictionary NOT NULL: `branch_id`, `branch_code`, `branch_name`, `branch_type`, `address`, `city`, `state`, `country`, `geographic_zone`, `phone`, `opening_time`, `closing_time`, `has_atms`, `has_teller_windows`, `branch_opening_date`, `branch_status`. Nullable: `postal_code`, `email`, `atm_count`, `teller_window_count`, `latitude`, `longitude`.

In [6]:
declared_not_null = [
    "branch_id", "branch_code", "branch_name", "branch_type", "address", "city", "state",
    "country", "geographic_zone", "phone", "opening_time", "closing_time", "has_atms",
    "has_teller_windows", "branch_opening_date", "branch_status",
]

rows = []
total = q("SELECT count(*) AS n FROM bronze.branches")["n"][0]
for col in declared_not_null:
    nulls = q(f"""
        SELECT count(*) AS n FROM bronze.branches
        WHERE {col} IS NULL OR trim(CAST({col} AS VARCHAR)) = ''
    """)["n"][0]
    rows.append({"column": col, "declared": "NOT NULL", "actual_nulls": nulls,
                 "null_rate": round(nulls / total, 4) if total else None})

pd.DataFrame(rows)

,column,declared,actual_nulls,null_rate
0,branch_id,NOT NULL,0,0.0
1,branch_code,NOT NULL,0,0.0
2,branch_name,NOT NULL,0,0.0
3,branch_type,NOT NULL,0,0.0
4,address,NOT NULL,0,0.0
5,city,NOT NULL,0,0.0
6,state,NOT NULL,0,0.0
7,country,NOT NULL,0,0.0
8,geographic_zone,NOT NULL,0,0.0
9,phone,NOT NULL,0,0.0


## 4. UNIQUE constraint audit -- `branch_code`

Declared `NOT NULL, UNIQUE` in the dictionary. Not currently declared in `contracts.py` at all (unlike `document_number`/`product_number`, which now are after the earlier fix) -- worth knowing whether it's actually clean before deciding whether to add it.

In [7]:
q("""
    SELECT
        count(*) AS total,
        count(DISTINCT branch_code) AS distinct_branch_code,
        count(*) FILTER (WHERE branch_code IS NULL OR trim(branch_code) = '') AS nulls
    FROM bronze.branches
""")

,total,distinct_branch_code,nulls
0,350,350,0


In [8]:
q("""
    SELECT branch_code, count(*) AS n, count(DISTINCT branch_id) AS distinct_branch_ids
    FROM bronze.branches
    GROUP BY 1 HAVING count(*) > 1
    ORDER BY n DESC
""")

,branch_code,n,distinct_branch_ids


## 5. Domain checks

In [9]:
# branch_type -- dictionary: Main, Express, Premium, Corporate
q("SELECT branch_type, count(*) AS n FROM bronze.branches GROUP BY 1 ORDER BY n DESC")

,branch_type,n
0,Express,130
1,Corporate,129
2,Premium,48
3,Main,43


In [10]:
# geographic_zone -- dictionary: Urban, Suburban, Rural
q("SELECT geographic_zone, count(*) AS n FROM bronze.branches GROUP BY 1 ORDER BY n DESC")

,geographic_zone,n
0,Urbana,350


In [11]:
# branch_status -- dictionary: Active, Temporarily Closed, Closed
q("SELECT branch_status, count(*) AS n FROM bronze.branches GROUP BY 1 ORDER BY n DESC")

,branch_status,n
0,Active,336
1,Temporarily Closed,14


## 6. Country canonicalization + state/country cross-check

Same pattern as `customers` -- `branches.country` also runs through `as_country()`.

In [12]:
print("Distinct raw country values in Bronze:")
display(q("SELECT country, count(*) AS n FROM bronze.branches GROUP BY 1 ORDER BY n DESC"))

print("Distinct country values in Silver:")
display(q("SELECT country, count(*) AS n FROM silver.dim_branches GROUP BY 1 ORDER BY n DESC"))

country_canonical = {
    "méxico": "México", "mexico": "México", "colombia": "Colombia", "argentina": "Argentina",
    "usa": "USA", "united states": "USA", "spain": "Spain", "brazil": "Brazil", "brasil": "Brazil",
}
raw_values = q("SELECT DISTINCT lower(trim(country)) AS v FROM bronze.branches WHERE country IS NOT NULL")["v"].tolist()
uncovered = [v for v in raw_values if v not in country_canonical]
print("Raw country values NOT covered by COUNTRY_CANONICAL:", uncovered or "none -- full coverage")

Distinct raw country values in Bronze:


,country,n
0,México,175
1,Colombia,105
2,Argentina,70


Distinct country values in Silver:


,country,n
0,México,175
1,Colombia,105
2,Argentina,70


Raw country values NOT covered by COUNTRY_CANONICAL: none -- full coverage


In [13]:
q("""
    SELECT country, state, count(*) AS n
    FROM silver.dim_branches
    GROUP BY 1, 2
    ORDER BY country, n DESC
""")

,country,state,n
0,Argentina,Córdoba,20
1,Argentina,Mendoza,16
2,Argentina,Buenos Aires,16
3,Argentina,Ciudad Autónoma de Buenos Aires,11
4,Argentina,Santa Fe,7
5,Colombia,Atlántico,24
6,Colombia,Bolívar,23
7,Colombia,Cundinamarca,22
8,Colombia,Valle del Cauca,21
9,Colombia,Antioquia,15


## 7. `postal_code` -- same VARCHAR-vs-numeric question as `customers`

In [14]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE TRY_CAST(postal_code AS BIGINT) IS NOT NULL) AS numeric_parseable,
        round(100.0 * count(*) FILTER (WHERE TRY_CAST(postal_code AS BIGINT) IS NOT NULL) / count(*), 2) AS pct_numeric,
        count(*) FILTER (WHERE postal_code IS NULL OR trim(postal_code) = '') AS nulls
    FROM bronze.branches
""")

,total,numeric_parseable,pct_numeric,nulls
0,350,280,80.0,0


## 8. `phone`/`email` shape sanity -- same treatment as `customers`' contact fields

In [15]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE phone IS NULL OR trim(phone) = '') AS phone_nulls,
        count(*) FILTER (WHERE phone IS NOT NULL AND trim(phone) <> ''
                          AND regexp_matches(phone, '[0-9]') = false) AS phone_no_digits,
        count(*) FILTER (WHERE email IS NULL OR trim(email) = '') AS email_nulls,
        count(*) FILTER (WHERE email IS NOT NULL AND trim(email) <> '' AND email NOT LIKE '%@%') AS email_missing_at
    FROM bronze.branches
""")

,total,phone_nulls,phone_no_digits,email_nulls,email_missing_at
0,350,0,0,0,0


## 9. `opening_time`/`closing_time` -- parsed correctly, and internally consistent

`branches_spec` casts both with `transform="time"`. Beyond a successful cast, closing should be after opening for every branch (no branch that "closes before it opens").

In [16]:
q("""
    SELECT
        typeof(opening_time), typeof(closing_time),
        count(*) FILTER (WHERE opening_time IS NULL) AS opening_nulls,
        count(*) FILTER (WHERE closing_time IS NULL) AS closing_nulls,
        count(*) FILTER (WHERE closing_time <= opening_time) AS closes_before_or_when_opens,
        min(opening_time) AS earliest_opening_time, max(opening_time) AS latest_opening_time,
        min(closing_time) AS earliest_closing_time, max(closing_time) AS latest_closing_time
    FROM silver.dim_branches
    GROUP BY ALL
""")

,typeof(opening_time),typeof(closing_time),opening_nulls,closing_nulls,closes_before_or_when_opens,earliest_opening_time,latest_opening_time,earliest_closing_time,latest_closing_time
0,TIME,TIME,0,0,0,08:00:00,09:30:00,17:00:00,20:00:00


## 10. ATM / teller-window flag-vs-count consistency

`has_atms`/`has_teller_windows` are booleans; `atm_count`/`teller_window_count` are the corresponding integers. Nothing in Silver enforces they agree -- a branch could have `has_atms=False` with a positive `atm_count`, or `has_atms=True` with `atm_count` NULL/0. Checking both directions.

In [17]:
print("Bronze raw has_atms values:")
display(q("SELECT typeof(has_atms), has_atms, count(*) AS n FROM bronze.branches GROUP BY ALL ORDER BY n DESC"))
print("Silver parsed has_atms values:")
display(q("SELECT typeof(has_atms), has_atms, count(*) AS n FROM silver.dim_branches GROUP BY ALL ORDER BY n DESC"))

Bronze raw has_atms values:


,typeof(has_atms),has_atms,n
0,VARCHAR,True,350


Silver parsed has_atms values:


,typeof(has_atms),has_atms,n
0,BOOLEAN,True,350


In [18]:
q("""
    SELECT
        has_atms,
        count(*) AS n,
        count(*) FILTER (WHERE atm_count IS NULL) AS atm_count_null,
        count(*) FILTER (WHERE atm_count = 0) AS atm_count_zero,
        count(*) FILTER (WHERE atm_count > 0) AS atm_count_positive,
        min(atm_count) AS min_atm_count, max(atm_count) AS max_atm_count
    FROM silver.dim_branches
    GROUP BY 1
    ORDER BY 1
""")

,has_atms,n,atm_count_null,atm_count_zero,atm_count_positive,min_atm_count,max_atm_count
0,True,350,0,0,350,2,8


In [19]:
print("Bronze raw has_teller_windows values:")
display(q("SELECT typeof(has_teller_windows), has_teller_windows, count(*) AS n FROM bronze.branches GROUP BY ALL ORDER BY n DESC"))
print("Silver parsed has_teller_windows values:")
display(q("SELECT typeof(has_teller_windows), has_teller_windows, count(*) AS n FROM silver.dim_branches GROUP BY ALL ORDER BY n DESC"))

Bronze raw has_teller_windows values:


,typeof(has_teller_windows),has_teller_windows,n
0,VARCHAR,True,350


Silver parsed has_teller_windows values:


,typeof(has_teller_windows),has_teller_windows,n
0,BOOLEAN,True,350


In [20]:
q("""
    SELECT
        has_teller_windows,
        count(*) AS n,
        count(*) FILTER (WHERE teller_window_count IS NULL) AS twc_null,
        count(*) FILTER (WHERE teller_window_count = 0) AS twc_zero,
        count(*) FILTER (WHERE teller_window_count > 0) AS twc_positive,
        min(teller_window_count) AS min_twc, max(teller_window_count) AS max_twc
    FROM silver.dim_branches
    GROUP BY 1
    ORDER BY 1
""")

,has_teller_windows,n,twc_null,twc_zero,twc_positive,min_twc,max_twc
0,True,350,0,0,350,3,12


## 11. `latitude`/`longitude` -- populated rate and plausible bounding box

México, Colombia, and Argentina together roughly span latitude -55 to 33 and longitude -118 to -34. Anything outside that box (or a suspicious (0, 0) "null island") is worth a second look rather than assuming the cast alone caught it.

In [21]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE latitude IS NOT NULL AND longitude IS NOT NULL) AS both_populated,
        min(latitude) AS min_lat, max(latitude) AS max_lat,
        min(longitude) AS min_lon, max(longitude) AS max_lon,
        count(*) FILTER (WHERE latitude NOT BETWEEN -56 AND 34) AS lat_out_of_box,
        count(*) FILTER (WHERE longitude NOT BETWEEN -119 AND -33) AS lon_out_of_box,
        count(*) FILTER (WHERE latitude = 0 AND longitude = 0) AS null_island
    FROM silver.dim_branches
""")

,total,both_populated,min_lat,max_lat,min_lon,max_lon,lat_out_of_box,lon_out_of_box,null_island
0,350,350,-34.690767,25.781772,-103.44855,0.099995,0,167,0


In [22]:
# Do the coordinates actually land inside their own declared country? A cheap proxy: each
# country's branches should cluster tightly, not spread across the whole bounding box.
q("""
    SELECT country,
        count(*) AS n,
        round(min(latitude), 2) AS min_lat, round(max(latitude), 2) AS max_lat,
        round(min(longitude), 2) AS min_lon, round(max(longitude), 2) AS max_lon
    FROM silver.dim_branches
    WHERE latitude IS NOT NULL AND longitude IS NOT NULL
    GROUP BY 1
""")

,country,n,min_lat,max_lat,min_lon,max_lon
0,México,175,-0.10,25.78,-103.45,0.1
1,Colombia,105,-0.10,6.34,-76.63,0.1
2,Argentina,70,-34.69,0.10,-64.28,0.1


## 11.5 Longitude anomaly follow-up -- 167/350 branches (47.7%) have an implausible longitude

Section 11 found `lon_out_of_box = 167` and every country's `max_lon` landing on the exact same value, `0.099995` -- a coincidence too precise to be random noise. If nearly half the branches share one specific wrong value (rather than being scattered errors), that points to a single sentinel/default longitude standing in for a real one, not independent corruption. These cells isolate exactly which rows and whether they share anything else in common.

In [23]:
# How many rows have exactly this value (or something extremely close to it, in case of float
# formatting noise), vs. how many are "out of box" but at some other bad value?
q("""
    SELECT
        count(*) FILTER (WHERE abs(longitude - 0.099995) < 0.0001) AS exact_sentinel_matches,
        count(*) FILTER (WHERE longitude NOT BETWEEN -119 AND -33 AND abs(longitude - 0.099995) >= 0.0001) AS other_out_of_box,
        count(*) FILTER (WHERE longitude NOT BETWEEN -119 AND -33) AS total_out_of_box
    FROM silver.dim_branches
""")

,exact_sentinel_matches,other_out_of_box,total_out_of_box
0,1,166,167


In [24]:
# Full detail on the affected rows -- looking for any shared trait (branch_type, opening date,
# _source_file, whether latitude also looks wrong) that would point at a root cause.
q_full("""
    SELECT branch_id, branch_code, branch_name, country, city, latitude, longitude, branch_type, branch_opening_date
    FROM silver.dim_branches
    WHERE longitude NOT BETWEEN -119 AND -33
    ORDER BY country, city
""")

,branch_id,branch_code,branch_name,country,city,latitude,longitude,branch_type,branch_opening_date
0,SUC-IPX8M3BR,S0348,Banco LATAM La Plata Centro,Argentina,La Plata,0.010832,-0.001430,Main,1990-06-25
1,SUC-RNIHGBDP,S0310,Banco LATAM La Plata 310,Argentina,La Plata,-0.020820,-0.038800,Express,2006-12-01
2,SUC-FXWYIOXF,S0286,Banco LATAM La Plata Centro,Argentina,La Plata,0.032035,0.028293,Main,2015-10-10
3,SUC-OJAQW0AJ,S0287,Banco LATAM La Plata 287,Argentina,La Plata,0.076195,-0.083144,Express,2014-02-09
4,SUC-ZSZKWMJL,S0289,Banco LATAM La Plata Plaza,Argentina,La Plata,0.016474,0.064569,Express,2006-03-17
5,SUC-5YFUX3Z6,S0340,Banco LATAM La Plata Centro,Argentina,La Plata,-0.076240,-0.020671,Corporate,2003-07-31
6,SUC-7HQXDYLE,S0291,Banco LATAM La Plata Plaza,Argentina,La Plata,-0.039348,-0.002815,Premium,1996-05-29
7,SUC-8433QXQ4,S0336,Banco LATAM La Plata Este,Argentina,La Plata,0.069424,0.095970,Corporate,2004-05-04
8,SUC-O09F6PWJ,S0320,Banco LATAM La Plata Centro,Argentina,La Plata,-0.099067,0.054482,Corporate,1999-01-19
9,SUC-7TUK3JBL,S0294,Banco LATAM La Plata Centro,Argentina,La Plata,-0.077071,-0.080133,Express,1997-07-18


In [25]:
# Is this isolated to longitude, or does latitude show the same pattern for these same rows?
# (i.e. did both coordinates get corrupted together, or just longitude?)
q("""
    SELECT
        count(*) AS affected_rows,
        min(latitude) AS min_lat_of_affected, max(latitude) AS max_lat_of_affected,
        count(*) FILTER (WHERE latitude NOT BETWEEN -56 AND 34) AS latitude_also_out_of_box
    FROM silver.dim_branches
    WHERE longitude NOT BETWEEN -119 AND -33
""")

,affected_rows,min_lat_of_affected,max_lat_of_affected,latitude_also_out_of_box
0,167,-0.099144,0.099491,0


In [26]:
# Bronze-level check -- is the bad value already wrong in Bronze (a source-data defect) or does
# it only appear wrong in Silver (a transform bug)? Compares raw text to the cast double.
q_full("""
    SELECT b.branch_id, b.longitude AS bronze_raw_longitude, s.longitude AS silver_cast_longitude
    FROM bronze.branches b
    JOIN silver.dim_branches s USING (branch_id)
    WHERE s.longitude NOT BETWEEN -119 AND -33
    ORDER BY b.branch_id
    LIMIT 20
""")

,branch_id,bronze_raw_longitude,silver_cast_longitude
0,SUC-033KHYS2,0.0704497,0.070450
1,SUC-0OJNMRLP,0.0644245,0.064424
2,SUC-12NKB9OB,0.0125987,0.012599
3,SUC-1C51KLB3,-0.092832,-0.092832
4,SUC-1G7XCTV8,0.0940801,0.094080
5,SUC-1PSI5W5X,-0.0263284,-0.026328
6,SUC-2L4DTLRM,0.0185063,0.018506
7,SUC-2NNWC3AV,-0.0627165,-0.062716
8,SUC-2QR0HN01,0.0561874,0.056187
9,SUC-2VHGDILI,-0.0745752,-0.074575


## 12. `branch_opening_date` sanity

In [27]:
q("""
    SELECT
        min(branch_opening_date) AS earliest_opening, max(branch_opening_date) AS latest_opening,
        count(*) FILTER (WHERE branch_opening_date > current_date) AS future_opening_dates,
        count(*) FILTER (WHERE branch_opening_date < DATE '2023-06-17') AS before_dataset_window,
        count(*) FILTER (WHERE branch_opening_date > DATE '2026-06-17') AS after_dataset_window
    FROM silver.dim_branches
""")

,earliest_opening,latest_opening,future_opening_dates,before_dataset_window,after_dataset_window
0,1990-01-03,2023-05-11,0,350,0


## 13. Known context

`branches.branch_id` is the parent side of the `registration_branch_id`/`assigned_branch_id` FK anomaly already fully diagnosed in `diagnose_branch_fk.ipynb` -- that investigation concluded the defect is in the *child* tables (`customers`, `service_agents`), not here. `products.opening_branch_id`, `transactions.branch_id`, and `complaints.related_branch_id`/`assigned_agent_id` were all confirmed to resolve cleanly against this table. Nothing in this notebook should change that conclusion; it's here for cross-reference only.

## Checklist

- [ ] Column inventory (1): spec/table agree
- [ ] Row reconciliation (2): 350 rows as the dictionary declares, no dedup mismatch
- [ ] NOT NULL audit (3): all 16 declared columns populated
- [ ] UNIQUE audit (4): `branch_code` genuinely unique, or note any real collision
- [ ] Domain checks (5): `branch_type`/`geographic_zone`/`branch_status` match declared sets
- [ ] Country canonicalization (6): full coverage, state/country pairs all plausible
- [ ] `postal_code` (7): note the numeric-parseable rate
- [ ] `phone`/`email` (8): note null rates and any structurally malformed values
- [ ] Time fields (9): no branch closes before/when it opens
- [ ] ATM/teller consistency (10): flag and count agree in both directions, or note where they don't
- [ ] Lat/long (11): no out-of-box coordinates, no null-island rows, each country's branches cluster sensibly
- [ ] `branch_opening_date` (12): no future dates, note any window mismatch (expected, per the `customers`/`products` precedent)

In [28]:
con.close()
print("closed")

closed
